Import libraries, set date range for analysis period, and initialize a `Sources` class object - it contains all the stock/index name and the corresponding tickers that we need. We also initialize `data_frames` to compile all DataFrames.

In [1]:
import requests
from datetime import datetime, timedelta
import pandas as pd
from fredapi import Fred 
import yfinance as yf
from dotenv import load_dotenv
import os
import sys

sys.path.append('..')
from modules.source import Sources, RAW_DATA_PATH
stock = Sources()

START = datetime(2015, 1, 1)
END = datetime(2026, 8, 6)
   
load_dotenv("../.env")
FRED_API_KEY = os.getenv('FRED_API_KEY')
if FRED_API_KEY is None: print('Enter your FRED API key in .env')
fred = Fred(FRED_API_KEY)

data_frames: dict[pd.DataFrame] = {}


1. **Fetch** — pull data from FRED
2. **Save** — write the raw response to `.csv`
3. **Process**
4. **Add to `data_frames` object**

We convert the raw data to DataFrame before saving it purely for pipeline consistency. Saving it as a Series instead would produce an identical CSV. The raw data is already clean, no further processing needed.

Since these operations are identical across all tickers, from here on we only note material deviations from this structure.

In [4]:
series = fred.get_series_first_release(stock.palm_oil_global.ticker)
df = series.to_frame(name=stock.palm_oil_global)
df.index.name = 'date'
df.index = pd.to_datetime(df.index)
df.to_csv(f"{RAW_DATA_PATH}/{stock.palm_oil_global}.csv")

df = df.loc[START:END]
data_frames[stock.palm_oil_global] = df

The EFFR data is adjusted for publication lag, so we shift it back to publication-aligned date, getting rid of information asymmetry.

In [5]:
series = fred.get_series(stock.EFFR.ticker)
df = series.to_frame(name=stock.EFFR)
df.index.name = 'date'

df.to_csv(f"{RAW_DATA_PATH}/{stock.EFFR}.csv")

lag = timedelta(days=1)
df = df.loc[START - lag : END - lag]
df = df.shift(1, lag)
data_frames[stock.EFFR] = df

We pull the data from yfinance, save it, and then flatten yfinance's multi-index columns (the Price/Ticker header levels aren't needed here), retain only the daily closing price.

In [6]:
yahoo_stocks = [stock.KLCI, stock.USDMYR, stock.brent_oil, stock.UST_10Y, stock.DXY, stock.VIX]

for name in yahoo_stocks:
   df = yf.download(name.ticker, START, END, progress=False)
   if df.empty:
      print(f"  FAILED - {name.ticker} returned no data, check the ticker code")
      continue
   
   df.to_csv(f'{RAW_DATA_PATH}/{name}.csv')
   if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.droplevel(1)

   print(f"{name} ({name.ticker}) - saved {len(df)} rows")
   df.columns.name = None
   df.index.name = 'date'

   df = df[['Close']].rename(columns={'Close': name})
   data_frames[name] = df

KLCI (^KLSE) - saved 2837 rows
USDMYR (MYR=X) - saved 3018 rows
Brent_Oil (BZ=F) - saved 2915 rows
UST_10Y (^TNX) - saved 2913 rows
DXY (DX-Y.NYB) - saved 2915 rows
VIX (^VIX) - saved 2915 rows


Now we fetch core CPI inflation data from DOSM, and replace the parquet's default numeric row index with `date`, since the numeric index carries no information.

Post-save, we narrow it down to overall division only (dropping the other divisions), and YoY inflation as our primary series - YoY strips out seasonal effects and matches how BNM and market commentary reference "inflation". Then adjust it back to publication-aligned date.

One thing to note: core CPI data is only available from 2018 onwards, so YoY (needing a 12-month lookback) is only available from 2019 onwards - this affects CPI-related analysis specifically, not the full project's date range.

In [7]:
df_cpi = pd.read_parquet('https://storage.dosm.gov.my/cpi/cpi_2d_core_inflation.parquet')

df_cpi['date'] = pd.to_datetime(df_cpi['date'])
df_cpi = df_cpi.set_index('date')

df_cpi.to_csv(f'{RAW_DATA_PATH}/CPI.csv')

df_cpi = df_cpi[df_cpi['division'] == 'overall']
df_cpi = df_cpi.drop(columns=['division', 'inflation_mom'])

df_cpi = df_cpi.shift(2, freq='MS')  # 2 months publication lag
df_cpi = df_cpi.loc[START:END]
df_cpi = df_cpi.dropna(how='any')
df_cpi = df_cpi.rename(columns={'inflation_yoy': stock.cpi_inflation_yoy})
data_frames[stock.cpi_inflation_yoy] = df_cpi

As with core CPI, for OPR, we set `date` as the index, then sort it before saving. The year-by-year fetch doesn't guarantee chronological order.

Post-save, we drop `year` (redundant once `date` is the index) and `change_in_opr` (we'll reconstruct this ourselves downstream), and rename `new_opr_level` to `OPR`.

In [8]:
headers = {'Accept': 'application/vnd.BNM.API.v1+json'}

records = []
for year in range(START.year, END.year + 1):
   resp = requests.get(f'https://api.bnm.gov.my/public/opr/year/{year}', headers=headers)
   records.extend(resp.json()['data'])

df_opr = pd.DataFrame(records)

df_opr['date'] = pd.to_datetime(df_opr['date'])
df_opr = df_opr.set_index('date')
df_opr = df_opr.sort_index()

df_opr.to_csv(f'{RAW_DATA_PATH}/{stock.OPR}.csv')

df_opr = df_opr.loc[START:END]

df_opr = df_opr.drop(columns=['year', 'change_in_opr'])
df_opr = df_opr.rename(columns={'new_opr_level': stock.OPR})

data_frames[stock.OPR] = df_opr

For each sector, we select 2-3 large, widely-held constituents (to reconstruct an equal-weighted index later on). We save to .csv after compiling companies - the compilation step has no loss of data, just cleaner.

In [ ]:
sector_constituents = {
   stock.financials: ['1155.KL', '1023.KL', '1295.KL'],   # Maybank, CIMB, Public Bank
   stock.plantation: ['5285.KL', '1961.KL', '2445.KL'],   # Sime Darby Plantation, IOI Corp, KLK
   stock.reits:      ['5227.KL', '5176.KL', '5235SS.KL'], # IGB REIT, Sunway REIT, KLCCP Stapled
   stock.technology: ['0166.KL', '0097.KL', '0128.KL'],   # Inari Amertron, ViTrox, Frontken
   stock.energy:     ['6033.KL', '5183.KL', '5681.KL'],   # PetGas, PetChem, Petronas Dagangan
   stock.industrial_products: ['8869.KL', '7113.KL'],     # Press Metal, Top Glove
}

for sector, tickers in sector_constituents.items():
   print(f"\n=== {sector.upper()} ===")
   closing_price = {}
   
   for t in tickers:
      df = yf.download(t, START, END, progress=False)
      if df.empty:
         print(f"  FAILED - {t} returned no data, check the ticker code")
         continue

      if isinstance(df.columns, pd.MultiIndex):
         df.columns = df.columns.droplevel(1)

      closing_price[t] = df['Close']
      print(f"  OK - {t} ({len(df)} rows)")

   sector_df = pd.DataFrame(closing_price)
   sector_df.index.name = 'date'
   sector_df.to_csv(f'{RAW_DATA_PATH}/sectors/{sector}.csv')
   data_frames[sector] = sector_df
   print(f"  Saved {sector}.csv - {sector_df.shape[1]} of {len(tickers)} tickers resolved")

In [ ]:
cutoff = datetime(2017, 11, 30) # 5285.KL (SD Guthrie Berhad) date of being listed
data_frames[stock.plantation] = data_frames[stock.plantation].loc[cutoff:]

us_market_ticker = [stock.EFFR, stock.UST_10Y, stock.USDMYR, stock.DXY, stock.VIX, stock.brent_oil, stock.palm_oil_global]

for i, df in data_frames.items():
   data_frames[i] = df.ffill()
   lag = timedelta(days=1)
   if i in us_market_ticker:
      data_frames[i] = df.shift(-1, lag).loc[START:]
   data_frames[i] = df.loc[START:END - lag]

In [20]:
for i, v in data_frames.items():
   print(f"{i}: {v.isna().sum()}")

Palm_Oil: Palm_Oil    0
dtype: int64
EFFR: EFFR    0
dtype: int64
KLCI: KLCI    0
dtype: int64
USDMYR: USDMYR    0
dtype: int64
Brent_Oil: Brent_Oil    0
dtype: int64
UST_10Y: UST_10Y    0
dtype: int64
DXY: DXY    0
dtype: int64
VIX: VIX    0
dtype: int64
cpi_inflation_yoy: cpi_inflation_yoy    0
dtype: int64
OPR: OPR    0
dtype: int64
financials: 1155.KL    0
1023.KL    0
1295.KL    0
dtype: int64
plantation: 5285.KL    0
1961.KL    0
2445.KL    0
dtype: int64
reits: 5227.KL      0
5176.KL      0
5235SS.KL    0
dtype: int64
technology: 0166.KL    0
0097.KL    0
0128.KL    0
dtype: int64
energy: 6033.KL    0
5183.KL    0
5681.KL    0
dtype: int64
industrial_products: 8869.KL    0
7113.KL    0
dtype: int64


non-synchronous trading / stale-price bias